## Lesson Overview

**What this lesson teaches:** how extended thinking appears in the Anthropic API, why some thinking is returned as a redacted block, and how to preserve API response blocks safely in a conversation.

**What's happening under the hood:**
1. Enable extended thinking and assign it a token budget.
2. Send Anthropic's special test string to deliberately produce redacted thinking.
3. Inspect the response as typed content blocks.
4. Extract only user-visible text for display.
5. Keep the complete assistant content when continuing the conversation.

**By the end:** you will be able to recognize and correctly handle `thinking`, `redacted_thinking`, and `text` blocks without attempting to expose private reasoning.

# Lesson 15: Handling Redacted Thinking

Extended thinking lets Claude spend additional tokens working through a problem before producing its visible answer. The API may encrypt part of that reasoning and return it as a `redacted_thinking` block. Applications should treat that block as opaque data: inspect its type, preserve it when required, and never try to decode or display it.

## The Response Shape

```text
API response
├── thinking block           → model reasoning returned by the API
├── redacted_thinking block  → opaque encrypted reasoning data
└── text block               → user-visible answer
```

A response can contain more than one block and the mix can vary. Code should branch on each block's `type` instead of assuming that `message.content[0]` contains displayable text.

## Setup

Install the dependencies once if needed:

```python
%pip install anthropic python-dotenv
```

Add `ANTHROPIC_API_KEY=...` to a `.env` file. The live example below sends an API request and may incur a small charge.

In [ ]:
import os

from anthropic import Anthropic
from anthropic.types import Message
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('ANTHROPIC_API_KEY')
client = Anthropic(api_key=api_key) if api_key else None
model = 'claude-sonnet-4-5'

if client is None:
    print('Setup incomplete: configure ANTHROPIC_API_KEY.')
else:
    print(f'Anthropic client ready; model: {model}')

## Conversation Helpers

The message helpers accept either plain content or an Anthropic `Message`. When an assistant response is added back to the history, its entire `content` list is retained. This matters because thinking and redacted-thinking blocks carry API state that should remain unchanged.

The `chat` helper adds the `thinking` parameter only when requested. Extended thinking uses a token budget within the response's overall `max_tokens` limit.

In [ ]:
def add_user_message(messages, message):
    content = message.content if isinstance(message, Message) else message
    messages.append({'role': 'user', 'content': content})


def add_assistant_message(messages, message):
    content = message.content if isinstance(message, Message) else message
    messages.append({'role': 'assistant', 'content': content})


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=None,
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    if client is None:
        raise RuntimeError('Configure ANTHROPIC_API_KEY before calling chat().')
    if thinking and thinking_budget < 1024:
        raise ValueError('thinking_budget must be at least 1024 tokens')

    params = {
        'model': model,
        'max_tokens': 4000,
        'messages': messages,
        'temperature': temperature,
        'stop_sequences': stop_sequences or [],
    }
    if thinking:
        params['thinking'] = {
            'type': 'enabled',
            'budget_tokens': thinking_budget,
        }
    if tools:
        params['tools'] = tools
    if system:
        params['system'] = system

    return client.messages.create(**params)


def text_from_message(message):
    return '\n'.join(
        block.text for block in message.content if block.type == 'text'
    )

## Trigger a Redacted-Thinking Block

Anthropic provides the following magic string specifically for testing this response path. It is not a prompt-injection technique and does not reveal reasoning; it asks the API to return a synthetic `redacted_thinking` block so an application can verify its handling logic.

In [ ]:
thinking_test_str = (
    'ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_'
    '46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB'
)

messages = []
add_user_message(messages, thinking_test_str)

response = None
if client is None:
    print('Skipping the live request. Complete setup, then rerun this cell.')
else:
    response = chat(messages, thinking=True, thinking_budget=1024)
    print('Response received.')

## Inspect Types, Not Private Data

For diagnostics, print the sequence of block types. Do not print a redacted block's `data` field: it is intentionally opaque and is not useful to an end user. The visible answer is assembled only from `text` blocks.

In [ ]:
if response is None:
    print('No response to inspect yet.')
else:
    block_types = [block.type for block in response.content]
    print('Block types:', block_types)
    print('Contains redacted thinking:', 'redacted_thinking' in block_types)
    print('\nVisible text:\n')
    print(text_from_message(response))

## Preserve the Full Assistant Response

Displaying and storing are different operations. Display only the text, but when continuing the same API conversation, append the assistant's complete structured content—including any thinking or redacted-thinking blocks—without editing or reordering it. Then add the next user turn.

In [ ]:
if response is None:
    print('No response to add to the conversation yet.')
else:
    add_assistant_message(messages, response)
    add_user_message(messages, 'Briefly summarize your visible answer.')
    print([message['role'] for message in messages])
    print('Assistant content preserved as structured blocks:',
          isinstance(messages[1]['content'], list))

## Local Sanity Checks

These checks make no API calls. A tiny stand-in message confirms that text extraction ignores non-text blocks and that assistant content remains structured when it is added to history.

In [ ]:
class FakeBlock:
    def __init__(self, block_type, text=None):
        self.type = block_type
        self.text = text


class FakeMessage:
    def __init__(self, content):
        self.content = content


fake_message = FakeMessage([
    FakeBlock('redacted_thinking'),
    FakeBlock('text', 'Safe visible answer.'),
])
assert text_from_message(fake_message) == 'Safe visible answer.'
assert [block.type for block in fake_message.content] == [
    'redacted_thinking', 'text'
]
print('All local checks passed.')

## Practice: Make the Handler Robust

Try these one at a time and predict the result before running:

1. **Inspect normal thinking:** replace the test string with a multi-step reasoning problem. Compare the returned block types without printing private block contents.
2. **Change the budget:** increase `thinking_budget` while keeping it below `max_tokens`. Observe usage and response quality.
3. **Multiple text blocks:** construct a fake message containing two text blocks. Confirm that `text_from_message` joins both in order.
4. **Continue the conversation:** send the prepared follow-up in `messages` with thinking enabled and print only its visible text.
5. **Build a safe logger:** record the model, stop reason, token usage, and block types while excluding thinking contents and redacted data.

## Summary

- Extended thinking is enabled with a token budget inside the response token limit.
- API responses contain typed content blocks; do not assume every block is text.
- `redacted_thinking` is opaque by design and should never be decoded or displayed.
- User interfaces should render `text` blocks and safely ignore private reasoning blocks.
- Conversation history should preserve the assistant's complete structured response unchanged.
- The official magic string tests redaction handling without revealing genuine model reasoning.